In [1]:
import mesa
import numpy as np
import matplotlib.pyplot as plt
from enum import Enum
import warnings
warnings.filterwarnings('ignore')

class AgentType(Enum):
    PASSIVE = 1
    NORMAL = 2
    AGGRESSIVE = 3

TYPE_COEFFICIENTS = {
    AgentType.PASSIVE: 0.3,
    AgentType.NORMAL: 0.7,
    AgentType.AGGRESSIVE: 1.0
}

print("V9 Crossover Experiment: READY")
print(f"Mesa version: {mesa.__version__}")

V9 Crossover Experiment: READY
Mesa version: 3.5.1


In [2]:
class QueueAgent(mesa.Agent):
    def __init__(self, model, agent_type):
        super().__init__(model)
        self.agent_type = agent_type
        self.type_coeff = TYPE_COEFFICIENTS[agent_type]
        self.urgency = np.random.uniform(0.3, 1.0)
        self.social_inhibition = np.random.uniform(0.2, 0.8)
        self.exited = False
        self.entry_time = None
        self.latency = None

    @property
    def behavior_score(self):
        base_score = (self.type_coeff * self.urgency /
                     self.social_inhibition)
        if self.pos:
            neighbors = self.model.grid.get_neighbors(
                self.pos, moore=True,
                include_center=False, radius=2
            )
            density_factor = 1.0 + (len(neighbors) * 0.05)
        else:
            density_factor = 1.0
        return base_score * density_factor

    def step(self):
        if self.exited or self.pos is None:
            return
        if self.entry_time is None:
            self.entry_time = self.model.steps
        x, y = self.pos
        move_prob = min(self.behavior_score, 1.0)
        if np.random.random() < move_prob:
            new_y = y - 1
            if new_y < 0:
                self.model.grid.remove_agent(self)
                self.exited = True
                self.latency = self.model.steps - self.entry_time
                self.model.exited_count += 1
                self.model.record_agent(self)
                return
            new_pos = (x, new_y)
            cell_contents = self.model.grid.get_cell_list_contents(
                [new_pos]
            )
            max_occupancy = (
                3 if self.agent_type == AgentType.AGGRESSIVE
                else 2 if self.agent_type == AgentType.NORMAL
                else 1
            )
            if len(cell_contents) < max_occupancy:
                self.model.grid.move_agent(self, new_pos)


class BunchQueueModel(mesa.Model):
    def __init__(self, n_agents=100, width=20, height=30,
                 pct_aggressive=0.15, pct_normal=0.25):
        super().__init__()
        self.width = width
        self.height = height
        self.steps = 0
        self.exited_count = 0
        self.total_agents = n_agents
        self.latencies = {
            AgentType.PASSIVE: [],
            AgentType.NORMAL: [],
            AgentType.AGGRESSIVE: []
        }
        self.grid = mesa.space.MultiGrid(
            width, height, torus=False
        )
        n_aggressive = int(n_agents * pct_aggressive)
        n_normal = int(n_agents * pct_normal)
        n_passive = n_agents - n_aggressive - n_normal
        agent_types = (
            [AgentType.AGGRESSIVE] * n_aggressive +
            [AgentType.NORMAL] * n_normal +
            [AgentType.PASSIVE] * n_passive
        )
        np.random.shuffle(agent_types)
        for agent_type in agent_types:
            agent = QueueAgent(self, agent_type)
            x = np.random.randint(0, width)
            y = np.random.randint(height // 2, height)
            self.grid.place_agent(agent, (x, y))

    def record_agent(self, agent):
        if agent.latency is not None:
            self.latencies[agent.agent_type].append(
                agent.latency
            )

    def step(self):
        self.steps += 1
        self.agents.shuffle_do("step")

    def run(self, max_steps=500):
        for _ in range(max_steps):
            self.step()
            if self.exited_count >= self.total_agents:
                break
        return self.exited_count

print("Agent and Model classes defined!")
print("Ready for crossover experiment")

Agent and Model classes defined!
Ready for crossover experiment


In [3]:
# V9 CROSSOVER EXPERIMENT
# Find where passive agent latency flips
# from benefit to harm as scale increases

print("V9 CROSSOVER EXPERIMENT")
print("="*60)
print("Testing passive agent latency across scales")
print("Aggression levels: 5% vs 30%")
print("="*60)

# Scale configurations
scale_configs = [
    {'name': '100',   'n': 100,  'w': 20,  'h': 30,  'steps': 500},
    {'name': '500',   'n': 500,  'w': 45,  'h': 65,  'steps': 1000},
    {'name': '1000',  'n': 1000, 'w': 65,  'h': 90,  'steps': 1500},
    {'name': '2500',  'n': 2500, 'w': 100, 'h': 140, 'steps': 2000},
    {'name': '5000',  'n': 5000, 'w': 140, 'h': 200, 'steps': 3000},
]

# Test two aggression levels
aggression_scenarios = [
    {'name': 'Orderly',    'pct_agg': 0.05, 'pct_norm': 0.20},
    {'name': 'Bunch 15%',  'pct_agg': 0.15, 'pct_norm': 0.25},
    {'name': 'High 30%',   'pct_agg': 0.30, 'pct_norm': 0.25},
]

n_runs = 20
crossover_results = {}

for scale in scale_configs:
    print(f"\nScale: {scale['name']} agents")
    print("-"*60)
    crossover_results[scale['name']] = {}

    for scenario in aggression_scenarios:
        passive_lats = []
        aggr_lats = []

        for run in range(n_runs):
            model = BunchQueueModel(
                n_agents=scale['n'],
                width=scale['w'],
                height=scale['h'],
                pct_aggressive=scenario['pct_agg'],
                pct_normal=scenario['pct_norm']
            )
            model.run(max_steps=scale['steps'])

            if model.latencies[AgentType.PASSIVE]:
                passive_lats.extend(
                    model.latencies[AgentType.PASSIVE]
                )
            if model.latencies[AgentType.AGGRESSIVE]:
                aggr_lats.extend(
                    model.latencies[AgentType.AGGRESSIVE]
                )

        avg_passive = np.mean(passive_lats) if passive_lats else 0
        avg_aggr = np.mean(aggr_lats) if aggr_lats else 0
        ratio = avg_passive / avg_aggr if avg_aggr > 0 else 0

        crossover_results[scale['name']][scenario['name']] = {
            'passive': avg_passive,
            'aggressive': avg_aggr,
            'ratio': ratio
        }

        print(f"  {scenario['name']:<12} "
              f"Passive: {avg_passive:6.1f}  "
              f"Aggr: {avg_aggr:6.1f}  "
              f"Ratio: {ratio:.2f}")

# Summary — find crossover
print("\n" + "="*60)
print("CROSSOVER SUMMARY")
print("Passive latency vs Orderly baseline")
print("="*60)
print(f"{'Scale':<8} {'Orderly':>10} "
      f"{'Bunch 15%':>10} {'High 30%':>10} "
      f"{'Verdict'}")
print("-"*60)

for scale in scale_configs:
    r = crossover_results[scale['name']]
    orderly = r['Orderly']['passive']
    bunch = r['Bunch 15%']['passive']
    high = r['High 30%']['passive']

    # Does aggression help or harm passive agents?
    if bunch < orderly:
        verdict = "HELPS"
    elif bunch > orderly * 1.1:
        verdict = "HARMS"
    else:
        verdict = "NEUTRAL"

    print(f"{scale['name']:<8} "
          f"{orderly:>10.1f} "
          f"{bunch:>10.1f} "
          f"{high:>10.1f} "
          f"{verdict}")

print("="*60)
print("\nExperiment complete!")

V9 CROSSOVER EXPERIMENT
Testing passive agent latency across scales
Aggression levels: 5% vs 30%

Scale: 100 agents
------------------------------------------------------------
  Orderly      Passive:  152.0  Aggr:   47.7  Ratio: 3.19
  Bunch 15%    Passive:  149.1  Aggr:   46.7  Ratio: 3.19
  High 30%     Passive:  153.7  Aggr:   47.2  Ratio: 3.26

Scale: 500 agents
------------------------------------------------------------
  Orderly      Passive:  357.7  Aggr:   99.9  Ratio: 3.58
  Bunch 15%    Passive:  348.8  Aggr:  101.2  Ratio: 3.45
  High 30%     Passive:  336.0  Aggr:  101.1  Ratio: 3.32

Scale: 1000 agents
------------------------------------------------------------
  Orderly      Passive:  519.6  Aggr:  142.0  Ratio: 3.66
  Bunch 15%    Passive:  503.1  Aggr:  141.5  Ratio: 3.55
  High 30%     Passive:  488.8  Aggr:  141.2  Ratio: 3.46

Scale: 2500 agents
------------------------------------------------------------
  Orderly      Passive:  855.4  Aggr:  221.0  Ratio: 3.87
 